# Run Simulation

End-to-end integratietest van de rescheduling simulatie op de Brusselse corridor.

**Pipeline:**
1. Laad data via `data/loader.py`
2. Configureer controller en trigger
3. Run simulatie
4. Analyseer resultaten

## 0. Imports en logging

In [1]:
import sys
import os

# Voeg projectroot toe aan Python path
sys.path.insert(0, '/Users/ddw/Desktop/Rescheduling')

# Verificeer
print(sys.path[0])
import logging
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from data.loader          import load_all
from controller.controller import Controller
from controller.triggers   import make_trigger
from simulation.simulator  import Simulator

# Logging instellen — INFO toont wichtige stappen, DEBUG toont alles
logging.basicConfig(
    level  = logging.INFO,
    format = '[%(levelname)s] %(name)s — %(message)s'
)
logger = logging.getLogger('notebook')
print('Imports OK')

/Users/ddw/Desktop/Rescheduling
Imports OK


## 1. Data laden

In [2]:
import pandas as pd
import os
from pathlib import Path

# Zet working directory naar projectroot
os.chdir(Path().resolve().parent)
print(f"Working directory: {Path().resolve()}")

N_FREIGHT = 300
df = pd.read_parquet(f'data/gold/combined/{N_FREIGHT}/combined_timetable.parquet')

print(f"Rijen geladen: {len(df)}")
probleem = df[df['EXIT_SECONDS'] <= df['ENTRY_SECONDS']]
print(f"Rijen met exit <= entry: {len(probleem)}")

Working directory: /Users/ddw/Desktop/Rescheduling
Rijen geladen: 11977
Rijen met exit <= entry: 0


In [3]:
from data.loader import load_trains, load_segments, load_timetable

trains    = load_trains(df)
segments  = load_segments(df)
timetable = load_timetable(df)

n_passenger = sum(1 for t in trains.values() if t.is_passenger)
n_freight   = sum(1 for t in trains.values() if t.is_freight)

print(f'Treinen:  {len(trains):>5}  ({n_passenger} passenger, {n_freight} freight)')
print(f'Segmenten:{len(segments):>5}')
print(f'Timetable:{len(timetable):>5} (train, segment) combinaties')

[INFO] data.loader — Treinen geladen: 1319
[INFO] data.loader — Segmenten geladen: 185
[INFO] data.loader — Timetable geladen: 11977 (train_no, segment_id) combinaties


Treinen:   1319  (1019 passenger, 300 freight)
Segmenten:  185
Timetable:11977 (train, segment) combinaties


## 2. Controller configureren

Kies een trigger strategie: `periodic`, `event_driven` of `hybrid`.

In [4]:
# Periodic trigger — eenvoudigste strategie, roept solver elke 15 min aan
trigger = make_trigger(
    'periodic',
    periodic_freq = 900,  # elke 15 minuten
)

controller = Controller(
    trigger   = trigger,
    trains    = trains,
    segments  = segments,
    timetable = timetable,
)

print(f'Controller: {controller}')
print(f'Trigger:    {trigger}')

Controller: Controller(trigger=PeriodicTrigger(periodic_freq=900s), rescheduled=0, fcfs=0, skipped=0)
Trigger:    PeriodicTrigger(periodic_freq=900s)


## 3. Simulatie runnen

In [5]:
SEED = 42

simulator = Simulator(
    trains     = trains,
    segments   = segments,
    timetable  = timetable,
    controller = controller,
    seed       = SEED,
)

print(f'Simulator aangemaakt: {simulator}')
print('Simulatie starten...')

t0    = time.time()
state = simulator.run()
elapsed = time.time() - t0

print(f'\nSimulatie klaar in {elapsed:.1f}s')
print(state.summary())

[INFO] simulation.simulator — Queue geïnitialiseerd: 1319 events voor 1319 treinen


Simulator aangemaakt: Simulator(treinen=1319, queue=0 events, state=SystemState(t=0s | actief=0, klaar=0, wachtend=1319), dispatcher=Dispatcher(bezet=0 segmenten, wachtend=0 treinen))
Simulatie starten...
[t=5s] TRIGGERED — building instance...
Set parameter Username


[INFO] gurobipy — Set parameter Username


Set parameter LicenseID to value 2811974


[INFO] gurobipy — Set parameter LicenseID to value 2811974


Academic license - for non-commercial use only - expires 2027-04-22


[INFO] gurobipy — Academic license - for non-commercial use only - expires 2027-04-22
[INFO] data.running_distributions — Rijtijdverdelingen geladen: 103 secties


[t=5s] RESCHEDULED — status=optimal, objective=0.00, solver_runtime=0.00s
[t=60s] SKIPPED — trigger did not fire
[t=95s] SKIPPED — trigger did not fire
[t=153s] SKIPPED — trigger did not fire
[t=180s] SKIPPED — trigger did not fire
[t=185s] SKIPPED — trigger did not fire
[t=189s] SKIPPED — trigger did not fire
[t=194s] SKIPPED — trigger did not fire
[t=240s] SKIPPED — trigger did not fire
[t=251s] SKIPPED — trigger did not fire
[t=287s] SKIPPED — trigger did not fire
[t=298s] SKIPPED — trigger did not fire
[t=302s] SKIPPED — trigger did not fire
[t=305s] SKIPPED — trigger did not fire
[t=310s] SKIPPED — trigger did not fire
[t=355s] SKIPPED — trigger did not fire
[t=360s] SKIPPED — trigger did not fire
[t=365s] SKIPPED — trigger did not fire
[t=411s] SKIPPED — trigger did not fire
[t=426s] SKIPPED — trigger did not fire
[t=429s] SKIPPED — trigger did not fire
[t=482s] SKIPPED — trigger did not fire
[t=485s] SKIPPED — trigger did not fire
[t=508s] SKIPPED — trigger did not fire
[t=550s]

KeyboardInterrupt: 

## 4. Resultaten analyseren

### 4.1 Bouw resultaten DataFrame

In [ ]:
records = []

for train_id, train in trains.items():
    for seg_id in train.path:
        try:
            actual_entry = state.actual_entry(train_id, seg_id)
            actual_exit  = state.actual_exit( train_id, seg_id)
        except KeyError:
            continue

        planned_entry = timetable.scheduled_arrival(  train_id, seg_id)
        planned_exit  = timetable.scheduled_departure(train_id, seg_id)

        records.append({
            'train_id':     train_id,
            'train_type':   train.train_type.value,
            'train_subtype': train.train_subtype.value,
            'segment_id':   seg_id,
            'planned_entry': planned_entry,
            'planned_exit':  planned_exit,
            'actual_entry':  actual_entry,
            'actual_exit':   actual_exit,
            'entry_delay':   max(0.0, actual_entry - planned_entry),
            'exit_delay':    max(0.0, actual_exit  - planned_exit),
        })

df = pd.DataFrame(records)
print(f'Resultaten: {len(df)} rijen, {df["train_id"].nunique()} treinen')
df.head()

### 4.2 Samenvatting vertragingen

In [ ]:
# Vertraging per trein (op laatste segment)
final_delays = []
for train_id, train in trains.items():
    if state.is_finished(train_id):
        delay = state.current_delay(train_id)
        final_delays.append({
            'train_id':     train_id,
            'train_type':   train.train_type.value,
            'train_subtype': train.train_subtype.value,
            'final_delay_s': delay,
            'final_delay_min': delay / 60,
        })

df_delays = pd.DataFrame(final_delays)

print('=== Eindvertraging per treintype ===')
print(df_delays.groupby('train_subtype')['final_delay_min'].agg(['mean', 'median', 'max', 'count']).round(2))
print()
print('=== Overall ===')
print(f"  Treinen op tijd (≤0s):     {(df_delays['final_delay_s'] == 0).sum():>4}")
print(f"  Treinen vertraagd (<5min): {((df_delays['final_delay_s'] > 0) & (df_delays['final_delay_s'] < 300)).sum():>4}")
print(f"  Treinen vertraagd (≥5min): {(df_delays['final_delay_s'] >= 300).sum():>4}")
print(f"  Gemiddelde vertraging:     {df_delays['final_delay_min'].mean():.2f} min")
print(f"  Maximale vertraging:       {df_delays['final_delay_min'].max():.2f} min")

### 4.3 Visualisatie: verdeling eindvertragingen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Links: histogram van eindvertragingen
ax = axes[0]
for subtype, group in df_delays.groupby('train_subtype'):
    delays = group['final_delay_min']
    ax.hist(delays, bins=30, alpha=0.6, label=subtype, density=True)
ax.set_xlabel('Eindvertraging (minuten)')
ax.set_ylabel('Densiteit')
ax.set_title('Verdeling eindvertragingen per treintype')
ax.legend()
ax.axvline(5, color='red', linestyle='--', linewidth=1, label='5 min drempel')

# Rechts: CDF van eindvertragingen
ax = axes[1]
for subtype, group in df_delays.groupby('train_subtype'):
    delays = group['final_delay_min'].sort_values()
    cdf    = np.arange(1, len(delays) + 1) / len(delays)
    ax.plot(delays, cdf, label=subtype)
ax.set_xlabel('Eindvertraging (minuten)')
ax.set_ylabel('Cumulatieve kans')
ax.set_title('CDF eindvertragingen')
ax.legend()
ax.axvline(5, color='red', linestyle='--', linewidth=1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.4 Visualisatie: vertraging over de dag

In [ ]:
# Vertraging per uur van de dag
df['hour'] = (df['planned_entry'] % 86400) / 3600
df['hour_bin'] = df['hour'].astype(int)

hourly = df.groupby(['hour_bin', 'train_type'])['exit_delay'].mean().reset_index()
hourly['exit_delay_min'] = hourly['exit_delay'] / 60

fig, ax = plt.subplots(figsize=(14, 5))
for train_type, group in hourly.groupby('train_type'):
    ax.plot(group['hour_bin'], group['exit_delay_min'], marker='o', label=train_type)

ax.set_xlabel('Uur van de dag')
ax.set_ylabel('Gemiddelde exitvertraging (min)')
ax.set_title('Gemiddelde vertraging per uur')
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(True, alpha=0.3)

# Periodes aanduiden
for start, end, label in [(6,9,'Morning Peak'), (16,19,'Evening Peak')]:
    ax.axvspan(start, end, alpha=0.1, color='orange', label=label)

plt.tight_layout()
plt.show()

### 4.5 Controller samenvatting

In [ ]:
summary = controller.summary()
print('=== Controller activiteit ===')
for key, val in summary.items():
    print(f'  {key:<20}: {val}')

### 4.6 Sanity checks

In [ ]:
n_finished  = sum(1 for t_id in trains if state.is_finished(t_id))
n_total     = len(trains)
pct_finished = n_finished / n_total * 100

print(f'Treinen voltooid: {n_finished}/{n_total} ({pct_finished:.1f}%)')

# Check: geen enkel segment heeft overlappende bezetting
overlaps = 0
for seg_id in segments:
    entries = [
        (state.actual_entry(t_id, seg_id), state.actual_exit(t_id, seg_id))
        for t_id in trains
        if seg_id in trains[t_id].path
        and seg_id in df[df['train_id'] == t_id]['segment_id'].values
    ]
    entries = [(e, x) for e, x in entries if e is not None and x is not None]
    entries.sort()
    for i in range(len(entries) - 1):
        if entries[i][1] > entries[i+1][0] + 0.001:
            overlaps += 1

print(f'Segmentoverlappen: {overlaps}  (verwacht: 0)')

# Check: simulatietijd consistent
print(f'Eindtijd simulatie: {state.current_time:.0f}s ({state.current_time/3600:.2f}u)')

## 5. Opslaan resultaten (optioneel)

In [ ]:
# Uncomment om resultaten op te slaan
# df.to_parquet('data/results/simulation_results.parquet', index=False)
# df_delays.to_parquet('data/results/final_delays.parquet', index=False)
# print('Resultaten opgeslagen')